In [ ]:
!pip install smolagents 'smolagents[litellm]' requests playwright --quiet
!playwright install chromium --with-deps


In [ ]:
import os
import time
import re
from google.colab import userdata

# Load API keys
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    os.environ["NCBI_API_KEY"] = userdata.get("NCBI_API_KEY")
except Exception:
    pass

import requests
from typing import List
from smolagents import ToolCallingAgent, LiteLLMModel, tool

# Create directory for screenshots
os.makedirs("screenshots", exist_ok=True)


# ---------- Tools ----------

@tool
def search_clinical_trials(
    query: str,
    sponsor: str = None,
    phase: str = None,
    statuses: List[str] = None,
    max_results: int = 10,
) -> list:
    """Search ClinicalTrials.gov for trials matching the given criteria.

    Returns a list of trials with summary information. Use this to find candidate
    trials, then call get_clinical_trial on specific NCT IDs for full details.

    IMPORTANT - SEARCH STRATEGY:
    A single search rarely gives comprehensive coverage of a drug class or
    therapeutic area. Trials are registered with inconsistent terminology —
    some use the generic class name, some use specific drug names, some use
    development codes, some use sponsor names.

    For thorough coverage, run multiple searches and combine results:
      - The drug class or mechanism (e.g., 'KRAS G12C inhibitor')
      - Specific approved/known drug names (e.g., 'sotorasib', 'adagrasib')
      - Development codes when known (e.g., 'AMG 510', 'MRTX849')
      - Sponsor names for major players (e.g., sponsor='Amgen')

    A trial may appear in only one of these searches. Deduplicate by NCT ID
    after combining.

    Args:
        query: Search terms describing the condition, intervention, or topic.
        sponsor: Optional lead sponsor name to filter by.
        phase: Optional phase filter. One of 'PHASE1', 'PHASE2', 'PHASE3', 'PHASE4'.
        statuses: Optional list of recruitment statuses. To find all 'active' or
            'ongoing' trials, pass ['RECRUITING', 'ACTIVE_NOT_RECRUITING'].
        max_results: Maximum number of results to return (default 10, max 50).

    Returns:
        A list of dictionaries with nct_id, title, status, phase, lead_sponsor,
        conditions, and start_date for each matching trial.
    """
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "query.term": query,
        "pageSize": min(max_results, 50),
        "format": "json",
    }
    if sponsor:
        params["query.lead"] = sponsor
    advanced_filters = []
    if phase:
        advanced_filters.append(f"AREA[Phase]{phase}")
    if statuses:
        status_or = " OR ".join(f"AREA[OverallStatus]{s}" for s in statuses)
        advanced_filters.append(f"({status_or})" if len(statuses) > 1 else status_or)
    if advanced_filters:
        params["filter.advanced"] = " AND ".join(advanced_filters)

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    results = []
    for study in data.get("studies", []):
        protocol = study.get("protocolSection", {})
        identification = protocol.get("identificationModule", {})
        status_mod = protocol.get("statusModule", {})
        sponsor_mod = protocol.get("sponsorCollaboratorsModule", {})
        conditions_mod = protocol.get("conditionsModule", {})
        design_mod = protocol.get("designModule", {})
        results.append({
            "nct_id": identification.get("nctId"),
            "title": identification.get("briefTitle"),
            "status": status_mod.get("overallStatus"),
            "phase": design_mod.get("phases", []),
            "lead_sponsor": sponsor_mod.get("leadSponsor", {}).get("name"),
            "conditions": conditions_mod.get("conditions", []),
            "start_date": status_mod.get("startDateStruct", {}).get("date"),
        })
    return results


@tool
def get_clinical_trial(nct_id: str) -> dict:
    """Fetch detailed information about a specific clinical trial from
    ClinicalTrials.gov.

    DATE INTERPRETATION:
    - 'primary_completion_date' = expected primary analysis date (readout timing).
    - 'completion_date' = end of all follow-up (often years past primary readout).
    Use primary_completion_date for competitive timing analysis.

    Args:
        nct_id: The NCT identifier of the trial (e.g., 'NCT04267848').

    Returns:
        A dictionary containing trial details.
    """
    url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()
    protocol = data.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    description = protocol.get("descriptionModule", {})
    conditions = protocol.get("conditionsModule", {})
    design = protocol.get("designModule", {})
    arms = protocol.get("armsInterventionsModule", {})
    outcomes = protocol.get("outcomesModule", {})
    return {
        "nct_id": nct_id,
        "title": identification.get("officialTitle") or identification.get("briefTitle"),
        "status": status.get("overallStatus"),
        "phase": design.get("phases", []),
        "study_type": design.get("studyType"),
        "lead_sponsor": sponsor.get("leadSponsor", {}).get("name"),
        "conditions": conditions.get("conditions", []),
        "brief_summary": description.get("briefSummary"),
        "interventions": [
            {"type": i.get("type"), "name": i.get("name")}
            for i in arms.get("interventions", [])
        ],
        "primary_outcomes": [
            {"measure": o.get("measure"), "timeframe": o.get("timeFrame")}
            for o in outcomes.get("primaryOutcomes", [])
        ],
        "start_date": status.get("startDateStruct", {}).get("date"),
        "primary_completion_date": status.get("primaryCompletionDateStruct", {}).get("date"),
        "completion_date": status.get("completionDateStruct", {}).get("date"),
        "url": f"https://clinicaltrials.gov/study/{nct_id}",
    }


@tool
def search_pubmed_for_trial(nct_id: str, max_results: int = 5) -> list:
    """Search PubMed for publications associated with a specific clinical trial.

    Returns publication metadata (PMID, title, authors, journal, year).
    Note: this returns SUMMARY data only — not abstract text, not specific
    clinical numbers. If you need specific efficacy data (PFS, OS, response
    rates), do NOT make them up; report only what was retrieved.

    INTERPRETATION GUIDE:
    - Multiple results: trial likely has published primary or secondary
      analyses. Cite the most prominent (Lancet, NEJM, JCO, Lancet Oncology).
    - Zero results: trial has not yet reported in peer-reviewed literature
      ("No peer-reviewed publications identified" — this is a real finding).

    Args:
        nct_id: The NCT identifier of the trial.
        max_results: Maximum number of publications to return (default 5).

    Returns:
        A list of dictionaries with pmid, title, authors, journal, year,
        and pubmed_url. Empty list if no publications found.
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
    api_key = os.environ.get("NCBI_API_KEY")

    search_params = {"db": "pubmed", "term": nct_id, "retmax": max_results, "retmode": "json"}
    if api_key:
        search_params["api_key"] = api_key
    search_response = requests.get(f"{base_url}/esearch.fcgi", params=search_params, timeout=30)
    search_response.raise_for_status()
    pmids = search_response.json().get("esearchresult", {}).get("idlist", [])
    if not pmids:
        time.sleep(1.0)
        return []

    time.sleep(1.0)

    summary_params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    if api_key:
        summary_params["api_key"] = api_key
    summary_response = requests.get(f"{base_url}/esummary.fcgi", params=summary_params, timeout=30)
    summary_response.raise_for_status()
    summary_data = summary_response.json()

    results = []
    for pmid in pmids:
        record = summary_data.get("result", {}).get(pmid, {})
        if not record:
            continue
        authors_list = record.get("authors", [])
        author_names = [a.get("name", "") for a in authors_list[:3]]
        pub_date = record.get("pubdate", "")
        year = pub_date.split()[0] if pub_date else ""
        results.append({
            "pmid": pmid,
            "title": record.get("title", ""),
            "authors": author_names,
            "journal": record.get("fulljournalname") or record.get("source", ""),
            "year": year,
            "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
        })
    time.sleep(1.0)
    return results


@tool
def screenshot_trial_page(nct_id: str) -> dict:
    """Capture a screenshot of a ClinicalTrials.gov trial page for visual evidence.

    Use this for trials whose claims are most important to back with visual
    proof — typically the must-include trials in your summary table, especially
    those with published results or distinctive trial designs (head-to-head
    comparisons, biomarker stratification, etc.).

    Don't screenshot every trial — that produces visual clutter without
    proportional trust gain. 3-5 screenshots is a good target for a CI report.

    The screenshot is saved as a PNG file in the screenshots/ directory and
    returns the file path. Reference the image in your Markdown report using:
        ![Trial NCTxxxxxxxx page](screenshots/NCTxxxxxxxx.png)

    Args:
        nct_id: The NCT identifier of the trial.

    Returns:
        A dictionary with:
        - 'success' (bool): whether the screenshot was captured.
        - 'path' (str): file path if successful (relative path for embedding).
        - 'url' (str): the page URL captured.
        - 'error' (str): error message if unsuccessful.
    """
    from playwright.sync_api import sync_playwright

    url = f"https://clinicaltrials.gov/study/{nct_id}"
    output_path = f"screenshots/{nct_id}.png"

    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            context = browser.new_context(viewport={"width": 1280, "height": 900})
            page = context.new_page()
            page.goto(url, wait_until="networkidle", timeout=30000)
            # Allow dynamic content (study details panel) to render
            page.wait_for_timeout(2000)
            # Capture just the visible viewport, not the full page (full page
            # screenshots are often too tall to be useful in reports)
            page.screenshot(path=output_path, full_page=False)
            browser.close()
        return {
            "success": True,
            "path": output_path,
            "url": url,
            "error": None,
        }
    except Exception as e:
        return {
            "success": False,
            "path": None,
            "url": url,
            "error": str(e),
        }


# ---------- Model & Agent ----------

model = LiteLLMModel(
    model_id="anthropic/claude-haiku-4-5",
    api_key=None,
    max_tokens=8000,
)

agent = ToolCallingAgent(
    tools=[
        search_clinical_trials,
        get_clinical_trial,
        search_pubmed_for_trial,
        screenshot_trial_page,
    ],
    model=model,
    max_steps=25,
)


# ---------- Run -------

def build_prompt(
    disease: str,
    drug_class: str,
    phase: str,
    known_drugs: List[str],
    known_codes: List[str] = None,
    major_sponsors: List[str] = None,
    worked_example: str = None,
) -> str:
    """Construct a CI agent prompt for any disease/drug-class/phase combination.

    Args:
        disease: e.g. "non-small cell lung cancer (NSCLC)" or "ovarian cancer"
        drug_class: e.g. "KRAS G12C inhibitors" or "PARP inhibitors"
        phase: e.g. "Phase 3", "Phase 2", "Phase 2 and Phase 3"
        known_drugs: Drug names to search by, e.g. ['olaparib', 'niraparib']
        known_codes: Optional development codes, e.g. ['AZD2281']
        major_sponsors: Optional sponsor names to search, e.g. ['AstraZeneca']
        worked_example: Optional concrete example showing one drug with multiple trials
    """
    drugs_str = ", ".join(known_drugs)
    codes_str = ", ".join(f"'{c}'" for c in known_codes) if known_codes else "(none specified)"
    sponsors_str = ", ".join(major_sponsors) if major_sponsors else "(unspecified — discover during search)"

    example_block = ""
    if worked_example:
        example_block = f"\nFor example, {worked_example}\n"

    return f"""Produce a competitive intelligence report on currently active
{phase} trials of {drug_class} in {disease}.

DEFINITIONS AND SCOPE:
- "Active" means trials that are either RECRUITING or ACTIVE_NOT_RECRUITING.
- A trial qualifies only if its enrolled population matches the indication
  AND the investigational drug belongs to the specified drug class (directly
  or as part of a combination).

SEARCH STRATEGY (CRITICAL):
Run multiple searches and combine the results before writing the report:
  1. The drug class: '{drug_class} {disease}'
  2. Each known drug by name: {drugs_str}
  3. Each known development code: {codes_str}
  4. By major sponsor: {sponsors_str}

INCLUDE ALL DISTINCT {phase} TRIALS PER DRUG:
Each drug typically has MULTIPLE distinct trials testing it in different
settings (lines of therapy, monotherapy vs combination, different populations).
Every distinct NCT ID matching the scope is a separate trial. Do not
deduplicate by drug.
{example_block}
PUBMED INTEGRATION:
For each significant trial, call search_pubmed_for_trial.
- If publications exist: cite as "Author et al., Journal Year (PMID: XXXXX)"
- If no publications: state "No peer-reviewed publications identified"

GROUNDING RULE (CRITICAL — DO NOT VIOLATE):
Every specific clinical claim must come from data your tools retrieved.
This includes efficacy numbers, hazard ratios, response rates, specific
dosing, biomarker cutoffs, regulatory dates. If a tool did not surface
a number, DO NOT INCLUDE IT. Acceptable substitutes:
  - "Primary results published" (without specific numbers)
  - "Demonstrated superiority over comparator on primary endpoint"
  - "Achieved primary endpoint"
search_pubmed_for_trial returns title/authors/journal/year — NOT abstract
content or efficacy data. Do not cite specific numbers from publications.

SCREENSHOT EVIDENCE:
For 3-5 of the most strategically important trials, call screenshot_trial_page
with the NCT ID. Embed using: ![Trial NCTxxxxxxxx](screenshots/NCTxxxxxxxx.png)

DELIVERABLE — three sections:

## Section 1: Summary Table
Columns: Sponsor | Drug | NCT ID | Status | Population/Setting | Comparator |
Primary Endpoint | Primary Completion Date | Publication Status.

## Section 2: Competitive Analysis
Concrete strategic analysis. Identify: who is in early-stage vs late-stage
development; who is monotherapy vs combination; what differentiates each
program; which readouts are timed to land first. Embed screenshots inline.

## Section 3: Limitations and Caveats
At least three bullet points: trials excluded with reasons, data fields
where ClinicalTrials.gov was insufficient, trials you suspect may exist
but did not surface.

FORMAT: Clean Markdown. Cite each trial's URL. Be concise. State
uncertainty rather than guessing."""


# ---------- Configure run ----------

# KRAS G12C in NSCLC
config_kras = {
    "disease": "non-small cell lung cancer (NSCLC)",
    "drug_class": "KRAS G12C inhibitors",
    "phase": "Phase 3",
    "known_drugs": [
        "sotorasib", "adagrasib", "divarasib", "olomorasib",
        "daraxonrasib", "opnurasib (JDQ443)", "D-1553",
    ],
    "known_codes": ["AMG 510", "MRTX849"],
    "major_sponsors": [
        "Amgen", "Mirati", "Bristol-Myers Squibb", "Hoffmann-La Roche",
        "Eli Lilly", "Novartis", "Revolution Medicines",
    ],
    "worked_example": (
        "sotorasib has at least two major Phase 3 NSCLC trials: "
        "CodeBreaK 200 (NCT04303780, 2L+ monotherapy vs docetaxel) and "
        "CodeBreaK 202 (NCT05920356, 1L combination vs immunotherapy). "
        "Both must appear separately."
    ),
}

# PARP inhibitors in ovarian cancer (different example to demonstrate generality)
config_parp = {
    "disease": "ovarian cancer",
    "drug_class": "PARP inhibitors",
    "phase": "Phase 3",
    "known_drugs": ["olaparib", "niraparib", "rucaparib", "talazoparib"],
    "known_codes": ["AZD2281"],
    "major_sponsors": [
        "AstraZeneca", "GSK", "Tesaro", "Clovis Oncology", "Pfizer",
    ],
    "worked_example": (
        "olaparib has multiple ovarian cancer Phase 3 trials across different "
        "settings — first-line maintenance, platinum-sensitive recurrent, BRCA-mutated, "
        "and HRD-positive populations. Include each distinct trial separately."
    ),
}

# Pick  run
config = config_kras  # change to config_parp to demonstrate generality

prompt = build_prompt(**config)
result = agent.run(prompt)

# Output filename includes the disease for clarity
disease_slug = config["drug_class"].lower().replace(" ", "_")
output_path = f"report_{disease_slug}.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(str(result))

print(f"\nReport saved to {output_path}")
print("\n" + "="*60)
print(result)

In [ ]:
# Read whatever the most recent run produced
config_for_eval = config_kras
disease_slug = config_for_eval["drug_class"].lower().replace(" ", "_")
output_path = f"report_{disease_slug}.md"

with open(output_path, "r", encoding="utf-8") as f:
    new_report = f.read()

scorecard = evaluate_report(new_report, GROUND_TRUTH)
print_scorecard(scorecard)

In [ ]:
import re

GROUND_TRUTH = {
    "must_include_trials": {
        "NCT04303780": {"drug": "sotorasib", "rationale": "CodeBreaK 200 — pivotal trial."},
        "NCT05132075": {"drug": "JDQ443", "rationale": "Novartis 2L+ trial."},
        "NCT06119581": {"drug": "olomorasib", "rationale": "SUNRAY-01."},
        "NCT06497556": {"drug": "divarasib", "rationale": "Head-to-head Roche trial."},
    },
    "should_include_trials": {
        "NCT06300177": {"drug": "D-1553"},
        "NCT06793215": {"drug": "divarasib"},
    },
    "must_not_include_trials": {
        "NCT06881784": {"drug": "daraxonrasib", "rationale": "Non-G12C primary endpoint."},
    },
    "required_sections": ["Summary Table", "Competitive Analysis", "Limitations"],
    "required_table_columns": [
        "Sponsor", "Drug", "NCT ID", "Status", "Line of Therapy",
        "Comparator", "Primary Endpoint",
    ],
}


def check_pmid_validity(report_text: str) -> dict:
    """Verify every PMID cited in the report exists in PubMed."""
    pmids = list(set(re.findall(r"PMID:?\s*(\d+)", report_text)))
    if not pmids:
        return {"total": 0, "valid": 0, "all_valid": True, "details": {},
                "no_pmids_found": True}

    details = {}
    for pmid in pmids:
        try:
            r = requests.get(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
                params={"db": "pubmed", "id": pmid, "retmode": "json"},
                timeout=10,
            )
            data = r.json().get("result", {}).get(pmid, {})
            details[pmid] = {
                "exists": bool(data and "title" in data),
                "title": data.get("title", "")[:80] if data else "",
                "journal": data.get("fulljournalname", "") if data else "",
            }
        except Exception as e:
            details[pmid] = {"exists": False, "title": "", "journal": "", "error": str(e)}
        time.sleep(0.4)

    valid_count = sum(1 for v in details.values() if v["exists"])
    return {
        "total": len(details),
        "valid": valid_count,
        "all_valid": valid_count == len(details),
        "details": details,
    }


def evaluate_report(report_text: str, ground_truth: dict) -> dict:
    results = {
        "must_include": {}, "should_include": {}, "must_not_include": {},
        "required_sections": {}, "required_columns": {},
        "behavioral": {}, "pmid_validity": {}, "summary": {},
    }

    for nct, info in ground_truth["must_include_trials"].items():
        present = nct in report_text
        drug_present = info["drug"].lower() in report_text.lower()
        results["must_include"][nct] = {
            "passed": present and drug_present,
            "drug": info["drug"],
            "rationale": info["rationale"],
        }

    for nct, info in ground_truth["should_include_trials"].items():
        results["should_include"][nct] = {
            "passed": nct in report_text,
            "drug": info["drug"],
        }

    for nct, info in ground_truth["must_not_include_trials"].items():
        present = nct in report_text
        properly_caveated = False
        if present:
            for match in re.finditer(re.escape(nct), report_text):
                window = report_text[max(0, match.start()-500):match.end()+500].lower()
                if any(w in window for w in ["exclud", "caveat", "limitation",
                                              "not include", "tangential", "non-g12c"]):
                    properly_caveated = True
                    break
        results["must_not_include"][nct] = {
            "passed": (not present) or properly_caveated,
            "present": present,
            "properly_caveated": properly_caveated,
        }

    for section in ground_truth["required_sections"]:
        results["required_sections"][section] = {"passed": section.lower() in report_text.lower()}
    for col in ground_truth["required_table_columns"]:
        results["required_columns"][col] = {"passed": col.lower() in report_text.lower()}

    text_lower = report_text.lower()
    results["behavioral"]["distinguishes_1l_vs_2l"] = {
        "passed": ("1l" in text_lower or "first-line" in text_lower)
                  and ("2l" in text_lower or "previously treated" in text_lower)
    }
    lim_match = re.search(r"(limitation|caveat).*?$", report_text, re.IGNORECASE | re.DOTALL)
    lim_section = lim_match.group(0) if lim_match else ""
    nct_in_limitations = len(re.findall(r"NCT\d+", lim_section))
    results["behavioral"]["limitations_substantive"] = {
        "passed": nct_in_limitations >= 2,
        "nct_mentions_in_limitations": nct_in_limitations,
    }
    pubmed_signals = ["pmid", "pubmed", "published", "publication", "lancet", "nejm",
                      "no peer-reviewed", "no publications"]
    pubmed_mentions = sum(1 for s in pubmed_signals if s in text_lower)
    results["behavioral"]["used_pubmed"] = {
        "passed": pubmed_mentions >= 2,
        "signals_found": pubmed_mentions,
    }

    # screenshot evidence check
    screenshot_count = len(re.findall(r"!\[.*?\]\(screenshots/", report_text))
    results["behavioral"]["used_screenshots"] = {
        "passed": screenshot_count >= 2,
        "screenshot_count": screenshot_count,
    }

    # PMID validity check
    pmid_check = check_pmid_validity(report_text)
    results["pmid_validity"] = pmid_check

    must_inc_passed = sum(1 for v in results["must_include"].values() if v["passed"])
    should_inc_passed = sum(1 for v in results["should_include"].values() if v["passed"])
    must_not_passed = sum(1 for v in results["must_not_include"].values() if v["passed"])
    sec_passed = sum(1 for v in results["required_sections"].values() if v["passed"])
    col_passed = sum(1 for v in results["required_columns"].values() if v["passed"])
    beh_passed = sum(1 for v in results["behavioral"].values() if v["passed"])

    results["summary"] = {
        "must_include": f"{must_inc_passed}/{len(results['must_include'])}",
        "should_include": f"{should_inc_passed}/{len(results['should_include'])}",
        "must_not_include_handled": f"{must_not_passed}/{len(results['must_not_include'])}",
        "sections": f"{sec_passed}/{len(results['required_sections'])}",
        "columns": f"{col_passed}/{len(results['required_columns'])}",
        "behavioral": f"{beh_passed}/{len(results['behavioral'])}",
        "pmid_validity": f"{pmid_check['valid']}/{pmid_check['total']}" if pmid_check['total'] > 0 else "no PMIDs",
        "critical_failures": (len(results["must_include"]) - must_inc_passed)
                            + (pmid_check['total'] - pmid_check['valid']),
    }
    return results


def print_scorecard(results: dict):
    print("="*70)
    print("EVAL SCORECARD")
    print("="*70)
    s = results["summary"]
    print(f"\n📊 Summary:")
    print(f"  Must-include trials:        {s['must_include']}")
    print(f"  Should-include trials:      {s['should_include']}")
    print(f"  Must-not-include handled:   {s['must_not_include_handled']}")
    print(f"  Required sections:          {s['sections']}")
    print(f"  Required columns:           {s['columns']}")
    print(f"  Behavioral checks:          {s['behavioral']}")
    print(f"  PMID validity:              {s['pmid_validity']}")
    print(f"  Critical failures:          {s['critical_failures']}")

    print(f"\n🔴 Critical failures (must-include trials missing):")
    crit_any = False
    for nct, info in results["must_include"].items():
        if not info["passed"]:
            crit_any = True
            print(f"  ❌ {nct} ({info['drug']}): {info['rationale']}")
    if not crit_any:
        print("  ✅ None")

    print(f"\n🟡 Should-include trials missing:")
    missing = [nct for nct, info in results["should_include"].items() if not info["passed"]]
    if missing:
        for nct in missing:
            print(f"  ⚠️  {nct} ({results['should_include'][nct]['drug']})")
    else:
        print("  ✅ All present")

    print(f"\n🟢 Must-not-include trials handled:")
    for nct, info in results["must_not_include"].items():
        if info["passed"]:
            tag = "present but properly caveated" if info["present"] else "correctly excluded"
            print(f"  ✅ {nct}: {tag}")
        else:
            print(f"  ❌ {nct}: present without proper caveat")

    print(f"\n📋 Section/column structure:")
    for section, info in results["required_sections"].items():
        print(f"  {'✅' if info['passed'] else '❌'} Section: {section}")
    for col, info in results["required_columns"].items():
        print(f"  {'✅' if info['passed'] else '❌'} Column: {col}")

    print(f"\n🧠 Behavioral checks:")
    for check, info in results["behavioral"].items():
        details = ""
        if "nct_mentions_in_limitations" in info:
            details = f" ({info['nct_mentions_in_limitations']} NCTs in limitations)"
        if "signals_found" in info:
            details = f" ({info['signals_found']} pubmed signals)"
        if "screenshot_count" in info:
            details = f" ({info['screenshot_count']} screenshots embedded)"
        print(f"  {'✅' if info['passed'] else '❌'} {check}{details}")

    print(f"\n📚 PMID validity:")
    pmid_check = results["pmid_validity"]
    if pmid_check.get("no_pmids_found"):
        print("  ⚠️  No PMIDs found in report — agent may not have cited any publications")
    elif pmid_check["all_valid"]:
        print(f"  ✅ All {pmid_check['total']} cited PMIDs verified in PubMed")
        for pmid, det in pmid_check["details"].items():
            print(f"     • PMID {pmid}: {det['journal']} — {det['title'][:60]}")
    else:
        print(f"  ❌ {pmid_check['total'] - pmid_check['valid']} of {pmid_check['total']} PMIDs are invalid (likely hallucinated)")
        for pmid, det in pmid_check["details"].items():
            status = "✅" if det["exists"] else "❌ NOT FOUND IN PUBMED"
            print(f"     • PMID {pmid}: {status}")
            if det["exists"]:
                print(f"        {det['journal']} — {det['title'][:60]}")

    print("\n" + "="*70)


# Run the evaluation
with open("report_kras_g12c_inhibitors.md", "r", encoding="utf-8") as f:
    new_report = f.read()

scorecard = evaluate_report(new_report, GROUND_TRUTH)
print_scorecard(scorecard)

In [ ]:
try:
    print(f"agent exists: {agent is not None}")
    print(f"config exists: {config['drug_class']}")
except NameError as e:
    print(f"State lost — need to rerun: {e}")
